# Visualizing Planetary Computer data with Lonboard

This notebook walks through interactive geospatial visualization with [Lonboard](https://developmentseed.org/lonboard/). Lonboard renders large vector datasets on a GPU-accelerated WebGL map directly in Jupyter. Key benefits:

1. **GPU rendering**: pan and zoom through *millions* of features without breaking interactivity.
2. **No tile server**: geometry streams to the browser as [Apache Arrow](https://arrow.apache.org/); there's no intermediate vector-tile service to stand up.
3. **Cloud-native vector**: read a STAC GeoParquet partition straight off Azure Blob into a `GeoDataFrame`.
4. **Composable**: stack multiple vector layers in one `Map`.
5. **Data-driven styling**: color features by an attribute and mutate the layer in place.

We'll render [Microsoft Building Footprints](https://planetarycomputer.microsoft.com/dataset/ms-buildings) over Portland, Oregon: hundreds of thousands of polygons in a single layer.

The companion [Lonboard tutorial](../overview/lonboard.md) has the full narrative.

## Install

In [ ]:
%pip install --quiet lonboard pystac-client planetary-computer geopandas deltalake adlfs mercantile

## Open the Planetary Computer STAC catalog

`modifier=planetary_computer.sign_inplace` signs every asset as the search returns, so the GeoParquet partition can be read directly.

**Expected result:** working `catalog` client, no output printed.

In [2]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

## Find the building-footprints partition for Portland

The `ms-buildings` collection is partitioned by [quadkey](https://learn.microsoft.com/en-us/bingmaps/articles/bing-maps-tile-system). Use `mercantile` to convert a Portland coordinate to a zoom-9 quadkey, then fetch the STAC item whose partition covers it.

**Expected result:** one matching item and its Delta Table `data` asset.

In [ ]:
import mercantile

tile = mercantile.tile(-122.66, 45.52, 9)
quadkey = mercantile.quadkey(*tile)

item = next(catalog.search(
    collections=["ms-buildings"],
    query={
        "msbuildings:region": {"eq": "UnitedStates"},
        "msbuildings:quadkey": {"eq": quadkey},
    },
).items())
asset = item.assets["data"]
quadkey, item.id

## Load the footprints into a GeoDataFrame

The asset is a Delta Table partition on Azure Blob. Open it with `deltalake`, enumerate the parquet files in the partition, then read each one with `geopandas`. Clip to the Portland metro for a focused view.

**Expected result:** a few hundred thousand building polygons.

In [ ]:
import geopandas as gpd
import pandas as pd
from deltalake import DeltaTable

storage_options = {
    "account_name": asset.extra_fields["table:storage_options"]["account_name"],
    "sas_token": asset.extra_fields["table:storage_options"]["credential"],
}
table = DeltaTable(asset.href, storage_options=storage_options)
gdf = pd.concat([
    gpd.read_parquet(uri, storage_options=storage_options)
    for uri in table.file_uris()
])
gdf = gdf.cx[-122.85:-122.45, 45.42:45.62]

len(gdf)

## Render the footprints

`PolygonLayer.from_geopandas()` uploads the geometry to the GPU as Arrow. The map below is fully interactive. Pan and zoom through every building with no tile server in the loop.

**Expected result:** an interactive map of Portland's building footprints.

In [5]:
from lonboard import Map, PolygonLayer

layer = PolygonLayer.from_geopandas(
    gdf,
    get_fill_color=[255, 140, 0, 170],
    get_line_color=[90, 40, 0],
    line_width_min_pixels=0.5,
)
m = Map(layer, view_state={"longitude": -122.66, "latitude": 45.52, "zoom": 12})
m

## Color by building height

Each footprint carries a `meanHeight`. Map it through a continuous colormap to shade every polygon: data-driven styling across the whole layer, evaluated on the GPU.

**Expected result:** the same footprints, now colored by height (`plasma`: purple = low, yellow = tall).

In [6]:
import matplotlib as mpl
from lonboard.colormap import apply_continuous_cmap

heights = gdf["meanHeight"].clip(0, 30)
normalized = (heights - heights.min()) / (heights.max() - heights.min())

layer.get_fill_color = apply_continuous_cmap(
    normalized.to_numpy(), mpl.colormaps["plasma"], alpha=0.8
)
m

## Mutate in place

Changing a layer property updates the existing map without re-uploading geometry.

**Expected result:** the rendered footprints redraw at 50% opacity.

In [7]:
layer.opacity = 0.5

## You're done

If every cell above rendered a map, the stack is wired up end-to-end: STAC search → cloud GeoParquet → Arrow upload → GPU-rendered vector → data-driven styling, all with no tile server.

Swap in your own bbox, collection, or `GeoDataFrame` and the same pattern applies. For pixel-level *raster* analysis (window reads, overview traversal), see the [async-geotiff tutorial](../overview/async-geotiff.md). For a standalone web app rather than a notebook, the [deck.gl-raster tutorial](../overview/deckgl-raster.md) builds a raster renderer in TypeScript.